# Forecast California Housing Prices

In [2]:
# Import libraries
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

# Dataset
from sklearn.datasets import fetch_california_housing

In [3]:
# Set random seed
np.random.seed(50)
torch.manual_seed(50)

In [4]:
# Load the dataset
data = fetch_california_housing()
features = pd.DataFrame(data.data, columns=data.feature_names)
targets = pd.Series(data.target, name='Price').values.astype(float).reshape(-1, 1)

**Questions**

* What is a `Bunch`?
* What is the default type for `data`?
* What is the default type for `target`?

**Answers**

1. A `Bunch` is a dictionary-like container provided by scikit-learn that supports both key-based and attribute-based access.
2. The default type for `data` is a NumPy array (`numpy.ndarray`).
3. The default type for `target` is a NumPy array (`numpy.ndarray`).

In [5]:
# Normalize features and target values
feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()
features_normalized = feature_scaler.fit_transform(features)
targets_normalized = target_scaler.fit_transform(targets)


def create_sequences(data, targets, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(targets[i + seq_length])
    return np.array(X), np.array(y)


seq_length = 20
X, y = create_sequences(features_normalized, targets_normalized, seq_length)

# Data cleaning and preprocessing

In [6]:
# Split the dataset into training (67%) and testing (33%) sets
train_size = int(len(X) * 0.67)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Convert NumPy arrays to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# LSTM model

In [7]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        output, _ = self.lstm(x)
        return self.fc(output[:, -1, :])


# Define model hyperparameters
input_size = X_train.shape[2]
hidden_size = 50
output_size = 1
num_layers = 2
learning_rate = 0.001
num_epochs = 50
l2_penalty = 0.01

# Instantiate the model, loss function, and optimizer
model = LSTMModel(input_size, hidden_size, output_size, num_layers)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=l2_penalty)

# GRU model

In [8]:
# Train the LSTM model
for epoch in range(num_epochs):
    optimizer.zero_grad()
    predictions = model(X_train_tensor)
    loss = criterion(predictions, y_train_tensor)
    loss.backward()
    optimizer.step()
    print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.6f}")

Epoch [1/50], Loss: 0.157260
Epoch [2/50], Loss: 0.145357
Epoch [3/50], Loss: 0.134166
Epoch [4/50], Loss: 0.123650
Epoch [5/50], Loss: 0.113777
Epoch [6/50], Loss: 0.104533
Epoch [7/50], Loss: 0.095921
Epoch [8/50], Loss: 0.087958
Epoch [9/50], Loss: 0.080678
Epoch [10/50], Loss: 0.074133
Epoch [11/50], Loss: 0.068384
Epoch [12/50], Loss: 0.063498
Epoch [13/50], Loss: 0.059535
Epoch [14/50], Loss: 0.056527
Epoch [15/50], Loss: 0.054465
Epoch [16/50], Loss: 0.053278
Epoch [17/50], Loss: 0.052829
Epoch [18/50], Loss: 0.052921
Epoch [19/50], Loss: 0.053326
Epoch [20/50], Loss: 0.053826
Epoch [21/50], Loss: 0.054245
Epoch [22/50], Loss: 0.054476
Epoch [23/50], Loss: 0.054483
Epoch [24/50], Loss: 0.054289
Epoch [25/50], Loss: 0.053957
Epoch [26/50], Loss: 0.053561
Epoch [27/50], Loss: 0.053174
Epoch [28/50], Loss: 0.052849
Epoch [29/50], Loss: 0.052623
Epoch [30/50], Loss: 0.052507
Epoch [31/50], Loss: 0.052497
Epoch [32/50], Loss: 0.052577
Epoch [33/50], Loss: 0.052722
Epoch [34/50], Loss

**Note:** The California housing dataset was sourced by scikit-learn from the StatLib repository: https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html

**Reference**  
Pace, R. K., & Barry, R. (1997). Sparse spatial autoregressions. Statistics & Probability Letters, 33(3), 291-297.

In [9]:
# Predict on the test set and invert the target scaling
model.eval()
with torch.no_grad():
    y_pred = model(X_test_tensor).cpu().numpy()

y_pred_inv = target_scaler.inverse_transform(y_pred)
y_test_inv = target_scaler.inverse_transform(y_test)

print("Predicted values:", y_pred_inv.squeeze())
mse = mean_squared_error(y_test_inv.squeeze(), y_pred_inv.squeeze())
print("Mean Squared Error:", mse)

Predicted values: [1.9351218 1.934201  1.9347136 ... 1.8713074 1.8653822 1.8600752]
Mean Squared Error: 1.4673422118614052


In [10]:
# Compare LSTM and GRU models with and without regularization
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        output, _ = self.gru(x)
        return self.fc(output[:, -1, :])


def train_and_evaluate(model_class, config):
    torch.manual_seed(50)
    experiment_model = model_class(
        input_size, config["hidden_size"], output_size, config["num_layers"]
    )
    experiment_optimizer = optim.Adam(
        experiment_model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )
    for _ in range(config["epochs"]):
        experiment_optimizer.zero_grad()
        output = experiment_model(X_train_tensor)
        experiment_loss = criterion(output, y_train_tensor)
        experiment_loss.backward()
        experiment_optimizer.step()

    experiment_model.eval()
    with torch.no_grad():
        predictions = experiment_model(X_test_tensor).numpy()
    predictions = target_scaler.inverse_transform(predictions).squeeze()
    actual = target_scaler.inverse_transform(y_test).squeeze()
    return mean_squared_error(actual, predictions)


baseline_config = {
    "hidden_size": 50,
    "num_layers": 2,
    "learning_rate": 0.001,
    "weight_decay": 0.0,
    "epochs": 30,
}

lstm_baseline_mse = train_and_evaluate(LSTMModel, baseline_config)
gru_baseline_mse = train_and_evaluate(GRUModel, baseline_config)
print("LSTM without regularization MSE:", lstm_baseline_mse)
print("GRU without regularization MSE:", gru_baseline_mse)

lstm_configs = [
    {"hidden_size": 32, "num_layers": 1, "learning_rate": 0.001, "weight_decay": 0.01, "epochs": 30},
    {"hidden_size": 64, "num_layers": 2, "learning_rate": 0.0005, "weight_decay": 0.001, "epochs": 30},
]
gru_configs = [
    {"hidden_size": 32, "num_layers": 1, "learning_rate": 0.001, "weight_decay": 0.01, "epochs": 30},
    {"hidden_size": 64, "num_layers": 2, "learning_rate": 0.0005, "weight_decay": 0.001, "epochs": 30},
]

lstm_tuning_mse = [train_and_evaluate(LSTMModel, config) for config in lstm_configs]
gru_tuning_mse = [train_and_evaluate(GRUModel, config) for config in gru_configs]

print("LSTM regularized configuration MSEs:", lstm_tuning_mse)
print("GRU regularized configuration MSEs:", gru_tuning_mse)

LSTM without regularization MSE: 1.5557179428784595
GRU without regularization MSE: 1.444004309264235
LSTM regularized configuration MSEs: [1.6265128180965918, 1.401019941803995]
GRU regularized configuration MSEs: [1.4704936625916012, 1.6029223057111777]


## GRU, regularization, and hyperparameter comparison

### No regularization

The baseline results were:

- LSTM without regularization: MSE = **1.5557**
- GRU without regularization: MSE = **1.4440**

Performance did not get worse in both models. The LSTM regularized baseline above had an MSE of **1.4673**, so removing regularization worsened the LSTM result. However, the no-regularization GRU result was better than both of the regularized GRU trials below. Regularization is important because it helps limit overfitting by discouraging overly complex parameter values, which can improve generalization and make training more efficient on unseen data. Excessive regularization can reduce performance by causing underfitting.

### Regularized hyperparameter trials

| Model | Hidden size | Layers | Learning rate | L2 penalty | Epochs | Test MSE |
|---|---:|---:|---:|---:|---:|---:|
| LSTM | 32 | 1 | 0.0010 | 0.010 | 30 | 1.6265 |
| LSTM | 64 | 2 | 0.0005 | 0.001 | 30 | **1.4010** |
| GRU | 32 | 1 | 0.0010 | 0.010 | 30 | **1.4705** |
| GRU | 64 | 2 | 0.0005 | 0.001 | 30 | 1.6029 |

The second LSTM configuration performed better than the original regularized LSTM (MSE **1.4673**), reducing the MSE to **1.4010**. Neither regularized GRU configuration improved on the no-regularization GRU baseline (MSE **1.4440**).